In [1]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import traceback

from aniposelib.cameras import CameraGroup
from pathlib import Path

from omegaconf import DictConfig

from lp3d_analysis.dataset_info import dataset_info
from lp3d_analysis.post_process import process_final_predictions, process_singleview_directory,setup_ensemble_dirs, get_original_structure
# from lp3d_analysis.post_process_concat import prepare_uncropped_csv_files 
from lp3d_analysis.io import load_cfgs
from lp3d_analysis.utils import generate_cropped_csv_file


from lightning_pose.utils import io as io_utils
from lightning_pose.utils.scripts import compute_metrics_single

from lightning_pose.api.model import Model

INFO:2026-03-26 13:57:20,323:jax._src.xla_bridge:927: Unable to initialize backend 'rocm': module 'jaxlib.xla_extension' has no attribute 'GpuAllocatorConfig'
INFO:jax._src.xla_bridge:Unable to initialize backend 'rocm': module 'jaxlib.xla_extension' has no attribute 'GpuAllocatorConfig'
INFO:2026-03-26 13:57:20,340:jax._src.xla_bridge:927: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
INFO:jax._src.xla_bridge:Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory


In [2]:
# user options
dataset_name = 'fly-anipose'
# dataset_name = 'chickadee-crop'
# dataset_name = 'ibl-mouse'
# dataset_name = 'two-mouse'
# dataset_name_chickadee = 'chickadee'
predictors = [
    # 'supervised_100_0-4/ensemble_mean',
    # 'supervised_100_0-4/ensemble_median',
    # 'supervised_100_0-4/eks_singleview',
    # 'supervised_100_0-4/eks_multiview_smooth_Matt',
    # 'supervised_100_0-4/eks_multiview_varinf_smooth',
    # 'supervised_200_0',
    # 'supervised_200_1',
    # 'supervised_200_2',
    # 'mvt_3d_loss_200_0',
    # 'mvt_3d_loss_200_2',
    # 'mvt_3d_loss_3218_0',
    # 'mvt_3d_loss_3221_0',
    # 'mvt_3d_loss_3110_0',
    # 'multiview_transformer_3235_0',
    # 'multiview_transformer_200_0-2/ensemble_median',
    # 'mvt_3d_loss_200_0-2/ensemble_median',
    'supervised_200_0-2/ensemble_median'
    # 'mvt_3d_loss_100_0-2/ensemble_median',
    # 'supervised_100_0-2/ensemble_median',
]
data_dir = f'/teamspace/studios/data/{dataset_name}'
# results_dir = f'/teamspace/studios/this_studio/outputs/{dataset_name}/test_200_MVT_3d_aug_patch_masking_new_labels'
# results_dir = f'/teamspace/studios/this_studio/outputs/{dataset_name}/test_200_multiview_resnet50'
# results_dir = f'/teamspace/studios/this_studio/outputs/{dataset_name}/test_200_multiview_resnet50'
# results_dir = f'/teamspace/studios/this_studio/outputs/{dataset_name}/test_200_MVT_3d_loss_patch_masking'
# results_dir = f'/teamspace/studios/this_studio/outputs/{dataset_name}/test_100_multiview_resnet50'
# results_dir = f'/teamspace/studios/this_studio/outputs/{dataset_name}/test_100_MVT_3d_loss_patch_masking'
# results_dir = f'/teamspace/studios/this_studio/outputs/{dataset_name}/test_3110_MVT_distil_patch_masking'
results_dir = f'/teamspace/studios/this_studio/outputs/{dataset_name}/test_200_dlc'

# extract dataset info
if dataset_name == 'fly-anipose':
    dataset_params = dataset_info[dataset_name]
elif dataset_name == 'chickadee-crop':
    dataset_params = dataset_info[dataset_name_chickadee]
elif dataset_name == 'ibl-mouse':
    dataset_params = dataset_info[dataset_name]
elif dataset_name == 'two-mouse':
    dataset_params = dataset_info[dataset_name]
cam_names = dataset_params['cam_names']
# cam_names = ['Cam-A', 'Cam-E']
n_keypoints = len(dataset_params['keypoints'])

# video_dir = 'videos-for-each-labeled-frame'
# video_dir = 'videos_new_fix'
video_dir = 'videos_new'
# video_dir = 'videos_new_add'
# video_dir = 'videos_new_segmentation'


if dataset_name == 'fly-anipose':
    # hard-code limbs to look at
    skeleton = {
        # outer limb
        'L1D-L1E': [3, 4],
        'L2D-L2E': [8, 9],
        'L3D-L3E': [13, 14],
        'L4D-L4E': [18, 19],
        'L5D-L5E': [23, 24],
        'L6D-L6E': [28, 29],
        # next outermost limb
        'L1C-L1D': [2, 3],
        'L2C-L2D': [7, 8],
        'L3C-L3D': [12, 13],
        'L4C-L4D': [17, 18],
        'L5C-L5D': [22, 23],
        'L6C-L6D': [27, 28],
    }
    files = os.listdir(Path(data_dir) / video_dir) # it was videos_new
    files_new = []
    for cam_name in cam_names:
        for file in files:
            if cam_name in file:
                files_new.append(file.replace(cam_name, '').replace('.mp4', ''))
    video_names = list(np.unique(files_new))
elif dataset_name == 'chickadee-crop':
    # raise NotImplementedError
    skeleton = {
        'topBeak-topHead': [0, 2],
        'leftEye-topHead': [8, 2],
        'rightEye-topHead': [13, 2],
        'topHead-backHead': [2, 3],
        'centerBack-backHead': [5, 3],
        'centerBack-centerChes': [5, 4],
        'centerBack-leftNeck': [5, 9],
        'leftNeck-leftWing': [9, 10],
        'centerBack-rightNeck': [5, 14],
        'rightNeck-rightWing': [14, 15],
        'baseTail-centerBack': [6, 5],
        'baseTail-tipTail': [6, 7],
        'baseTail-leftAnkle': [6, 11],
        'leftAnkle-leftFoot': [11, 12],
        'baseTail-rightAnkle': [6, 16],
        'rightAnkle-rightFoot': [16, 17],
    }
    files = os.listdir(Path(data_dir) / video_dir) # it was videos_new
    files_new = []
    for cam_name in cam_names:
        for file in files:
            if cam_name in file:
                files_new.append(file.replace(cam_name, '').replace('.mp4', ''))
    video_names = list(np.unique(files_new))
    # remove the last character fro, the video names

    if video_dir == 'videos-for-each-labeled-frame':
        video_names = [video_name[:-1] for video_name in video_names]
    else:
        # remove the bbox files from the video names
        video_names = [video_name for video_name in video_names if 'bbox' not in video_name]
        # if video_names include _.short
        video_names = [video_name.replace('_.short', '') for video_name in video_names]

elif dataset_name == 'ibl-mouse':
    files = os.listdir(Path(data_dir) / video_dir) # it was videos_new
    files_new = []
    for cam_name in cam_names:
        for file in files:
            if cam_name in file:
                files_new.append(file.replace(cam_name, '').replace('.mp4', ''))
    video_names = list(np.unique(files_new))
elif dataset_name == 'two-mouse':
    files = os.listdir(Path(data_dir) / video_dir) # it was videos_new
    files_new = []
    for cam_name in cam_names:
        for file in files:
            if cam_name in file:
                files_new.append(file.replace(cam_name, '').replace('.mp4', ''))
    video_names = list(np.unique(files_new))

In [3]:
video_names 

[np.str_('05272019_fly4_0_R1C24__rot-ccw-0.06_sec'),
 np.str_('05272019_fly4_0_R2C14__str-ccw-0.72_sec'),
 np.str_('05272019_fly4_0_R2C18__rot-cw-0.09_sec'),
 np.str_('05272019_fly4_0_R2C26__rot-ccw-0.18_sec'),
 np.str_('05272019_fly4_0_R3C16__rot-cw-0.03_sec'),
 np.str_('05272019_fly4_0_R3C26__rot-ccw-0.18_sec'),
 np.str_('05272019_fly5_0_R1C12__str-ccw-0.18_sec'),
 np.str_('05272019_fly5_0_R1C5__str-cw-0.18_sec'),
 np.str_('05272019_fly5_0_R2C16__rot-cw-0.03_sec'),
 np.str_('05272019_fly5_0_R2C5__str-cw-0.18_sec'),
 np.str_('05272019_fly5_0_R3C15__rot-cw-0_sec'),
 np.str_('05272019_fly5_0_R3C19__rot-cw-0.18_sec')]

Trying to take all of the csv files from videos-for-each-labeled-frame and run anipose 3D triangulation on them. If you remember any of the 3D stuff will only work in the full resolution space, not the cropped space (we had to do this mv eks for example).

I want to save the things in the reprojected folder inside the particular seed so then I can compute pixel error and all of this in the same way. I need to have the cropped files too so I can run pixel error calculation on that 

# For both full videos and short snippets

In [4]:
results_df = []
for predictor in predictors:
    print(f"creating anipose predictions for {predictor}")
    for video_name in video_names:
        print(f'video: {video_name}')
        calibration_file = Path(data_dir) / 'calibrations' / f'{video_name.replace("__", "_")}.toml' 
        if '.short' in calibration_file.name:
            calibration_file = calibration_file.with_name(calibration_file.name.replace('.short', ''))
        if '_.' in calibration_file.name:
            calibration_file = calibration_file.with_name(calibration_file.name.replace('_.', '.'))
        # if there is .short in the calibration file, remove it
        print(f'calibration file: {calibration_file}')
        assert calibration_file.is_file()
        
        # Check which directory structure we're dealing with
        # First, try the single file structure
        cam_files = {}
        all_files_exist = True
        single_file_structure = True
        
        for cam in cam_names:
            if dataset_name == 'fly-anipose':
                stem = video_name.replace("__", f"_{cam}_")
                data_path = Path(f'{results_dir}/{predictor}/{video_dir}') / f'{stem}.csv'
            elif dataset_name == 'chickadee-crop':
                # stem = f'{video_name}_{cam}'
                # data_path = Path(f'{results_dir}/{predictor}/{video_dir}') / f'{stem}.csv'
                stem = f'{video_name}_{cam}.short_uncropped'
                data_path = Path(f'{results_dir}/{predictor}/{video_dir}') / f'{stem}.csv'
            elif dataset_name == 'ibl-mouse':
                stem = video_name.replace("_.", f"_{cam}.")
                data_path = Path(f'{results_dir}/{predictor}/{video_dir}') / f'{stem}.csv'
            elif dataset_name == 'two-mouse':
                stem = video_name.replace("Defeat_", f"Defeat_{cam}")
                data_path = Path(f'{results_dir}/{predictor}/{video_dir}') / f'{stem}.csv'
            
            else:
                print(f"Unsupported dataset_name: {dataset_name}")
                all_files_exist = False
                break

            print(f'checking single file path: {data_path}')
            if data_path.is_file():
                cam_files[cam] = data_path
            else:
                all_files_exist = False
                break
        
        # If single file structure doesn't work, try snippet structure
        if not all_files_exist:
            print("Single file structure not found, trying snippet structure...")
            single_file_structure = False
            cam_dirs = {}
            all_dirs_exist = True
            
            for cam in cam_names:
                if dataset_name == 'fly-anipose':
                    data_path = Path(f'{results_dir}/{predictor}/{video_dir}') / f'{video_name.replace("__", f"_{cam}_")}'
                elif dataset_name == 'chickadee-crop':
                    data_path = Path(f'{results_dir}/{predictor}/{video_dir}') / f'{video_name}_{cam}'
                    print(f'data_path: {data_path}')
                
                # print(f'checking snippet directory path: {data_path}')
                if data_path.is_dir():
                    cam_dirs[cam] = data_path
                else:
                    all_dirs_exist = False
                    break
            
            if not all_dirs_exist:
                print(f"Skipping video {video_name} - neither structure found")
                continue
        else:
            print("Using single file structure")

        if single_file_structure:
            # Process single file structure
            snippet_files = [cam_files[cam_names[0]].name]
            print(f" snippet_files: {snippet_files}")
            
            for snippet_file in snippet_files:
                snippet_base = os.path.splitext(snippet_file)[0]
                print(f"Processing snippet: {snippet_base}")

                snippet_arrays = []
                all_snippets_valid = True
                template_df = None

                for cam in cam_names:
                    cam_snippet_path = cam_files[cam]
                    if not cam_snippet_path.exists():
                        print(f"Warning: Snippet file {cam_snippet_path} does not exist. Skipping.")
                        all_snippets_valid = False
                        break

                    try:
                        snippet_df = pd.read_csv(cam_snippet_path, index_col=0, header=[0, 1, 2])
                        if template_df is None:
                            template_df = snippet_df.copy()
                        xs = snippet_df.loc[:, snippet_df.columns.get_level_values('coords').isin(['x'])]
                        ys = snippet_df.loc[:, snippet_df.columns.get_level_values('coords').isin(['y'])]
                        snippet_arrays.append(np.stack([xs, ys], axis=2))
                    except Exception as e:
                        print(f"Error reading snippet file {cam_snippet_path}: {e}")
                        all_snippets_valid = False
                        break

                if not all_snippets_valid:
                    print(f"Skipping snippet {snippet_file}")
                    continue

                # Convert to numpy array and triangulate
                snippet_arrays = np.stack(snippet_arrays, axis=0)
                print(f'Snippet arrays shape: {snippet_arrays.shape}')

                camgroup = CameraGroup.load(calibration_file)
                print("Available methods:", [method for method in dir(camgroup) if 'triangulate' in method.lower()])

                snippet_points_3d = []
                for k in range(n_keypoints):
                    # print(f'triangulating keypoint {k}')
                    snippet_points_3d.append(camgroup.triangulate(snippet_arrays[:, :, k, :], fast=True))
                snippet_points_3d = np.stack(snippet_points_3d, axis=1)
                print(f'Snippet points_3d shape: {snippet_points_3d.shape}')

                for cam_idx, cam in enumerate(cam_names):
                    camera = camgroup.cameras[cam_idx]

                    points_flat = snippet_points_3d.reshape(-1, 3)
                    points_2d_flat = camera.project(points_flat)
                    n_frames = snippet_points_3d.shape[0]
                    points_2d = points_2d_flat.reshape(n_frames, n_keypoints, 2)

                    # this is for reprojection 
                    # Calculate reprojection error
                    original_2d = snippet_arrays[cam_idx]  # Original 2D points for this camera
                    reprojection_error = np.sqrt(np.sum((original_2d - points_2d)**2, axis=2))  # Euclidean distance per keypoint per frame
                    mean_error = np.mean(reprojection_error)
                    median_error = np.median(reprojection_error)
                    max_error = np.max(reprojection_error)
                    std_error = np.std(reprojection_error)

                    print(f"Camera {cam} reprojection error statistics:")
                    print(f"  Mean error: {mean_error:.4f} pixels")
                    print(f"  Median error: {median_error:.4f} pixels")
                    print(f"  Max error: {max_error:.4f} pixels")
                    print(f"  Std error: {std_error:.4f} pixels")
                    
                    # Optional: Print per-keypoint errors
                    keypoint_errors = np.mean(reprojection_error, axis=0)  # Average error per keypoint
                    print(f"  Per-keypoint mean errors: {keypoint_errors}")
                    
                    # Optional: Print per-frame errors
                    frame_errors = np.mean(reprojection_error, axis=1)  # Average error per frame
                    print(f"  Per-frame mean errors (first 10): {frame_errors[:10]}")

                    # Create ONLY the main reprojected directory (no camera subdirectories)
                    output_main_dir = Path(f'{results_dir}/{predictor}/{video_dir}_reprojected')
                    os.makedirs(output_main_dir, exist_ok=True)
                    print(f"Created main output directory: {output_main_dir}")

                    new_df = template_df.copy()

                    keypoints = new_df.columns.get_level_values('bodyparts').unique()
                    for i, kp in enumerate(keypoints):
                        kp_idx = dataset_params['keypoints'].index(kp)
                        kp_x_cols = new_df.columns[
                            (new_df.columns.get_level_values('bodyparts') == kp) &
                            (new_df.columns.get_level_values('coords') == 'x')
                        ]
                        kp_y_cols = new_df.columns[
                            (new_df.columns.get_level_values('bodyparts') == kp) &
                            (new_df.columns.get_level_values('coords') == 'y')
                        ]
                        for col in kp_x_cols:
                            new_df[col] = points_2d[:, kp_idx, 0]
                        for col in kp_y_cols:
                            new_df[col] = points_2d[:, kp_idx, 1]

                    # Save CSV file directly in main reprojected directory
                    out_name = cam_files[cam].name
                    snippet_output_file = output_main_dir / out_name
                    new_df.to_csv(snippet_output_file)
                    print(f"Saved reprojected 2D points to {snippet_output_file}")

                    # Construct bbox path for full videos
                    # For chickadee-crop full videos, bbox files are in videos_new_fix directory
                    # Pattern: {video_name}_{cam}.short_bbox.csv
                    bbox_path = Path(data_dir) / "videos_new_fix" / f"{video_name}_{cam}.short_bbox.csv"

                    if bbox_path.exists():
                        # Create cropped version
                        cropped_file = snippet_output_file.with_stem(snippet_output_file.stem.replace("_uncropped", ""))
                        generate_cropped_csv_file(
                            input_csv_file=str(snippet_output_file),
                            input_bbox_file=str(bbox_path),
                            output_csv_file=str(cropped_file),
                            img_height=320,
                            img_width=320,
                            mode="subtract"
                        )
                        print(f"Created cropped file: {cropped_file}")
                    else:
                        print(f"No bbox file found at {bbox_path}, skipping cropped file creation")

                    

        else:
            # Process snippet structure
            first_cam = cam_names[0]
            if dataset_name == 'fly-anipose':
                snippet_files = sorted([f for f in os.listdir(cam_dirs[first_cam]) if f.endswith('.csv')])
            elif dataset_name == 'chickadee-crop':
                snippet_files = sorted([f for f in os.listdir(cam_dirs[first_cam]) 
                                    if f.endswith('uncropped.csv')])

            # process each snippet individually
            for snippet_file in snippet_files:
                snippet_base = os.path.splitext(snippet_file)[0]
                print(f"Processing snippet: {snippet_base}")
                
                snippet_arrays = []
                all_snippets_valid = True
                template_df = None

                for cam in cam_names:
                    cam_snippet_path = cam_dirs[cam] / snippet_file
                    if not cam_snippet_path.exists():
                        print(f"Warning: Snippet file {cam_snippet_path} does not exist. Skipping.")
                        all_snippets_valid = False
                        break
                    
                    try:
                        snippet_df = pd.read_csv(cam_snippet_path, index_col=0, header=[0, 1, 2])
                        if template_df is None:
                            template_df = snippet_df.copy()
                        xs = snippet_df.loc[:, snippet_df.columns.get_level_values('coords').isin(['x'])]
                        ys = snippet_df.loc[:, snippet_df.columns.get_level_values('coords').isin(['y'])]
                        snippet_arrays.append(np.stack([xs, ys], axis=2))
                    except Exception as e:
                        print(f"Error reading snippet file {cam_snippet_path}: {e}")
                        all_snippets_valid = False
                        break
                
                if not all_snippets_valid:
                    print(f"Skipping snippet {snippet_file}")
                    continue

                # Convert to numpy array and triangulate
                snippet_arrays = np.stack(snippet_arrays, axis=0)
                print(f'Snippet arrays shape: {snippet_arrays.shape}')
                
                camgroup = CameraGroup.load(calibration_file)
                snippet_points_3d = []
                for k in range(n_keypoints):
                    snippet_points_3d.append(camgroup.triangulate(snippet_arrays[:, :, k, :], fast=True))
                
                snippet_points_3d = np.stack(snippet_points_3d, axis=1)
                print(f'Snippet points_3d shape: {snippet_points_3d.shape}')

                for cam_idx, cam in enumerate(cam_names):
                    camera = camgroup.cameras[cam_idx]
                    points_flat = snippet_points_3d.reshape(-1, 3)
                    points_2d_flat = camera.project(points_flat)
                    n_frames = snippet_points_3d.shape[0]
                    points_2d = points_2d_flat.reshape(n_frames, n_keypoints, 2)

                    # Create output directory for this camera
                    if dataset_name == 'fly-anipose':
                        output_cam_dir = Path(f'{results_dir}/{predictor}/{video_dir}_reprojected') / f'{video_name.replace("__", f"_{cam}_")}'
                    elif dataset_name == 'chickadee-crop':
                        output_cam_dir = Path(f'{results_dir}/{predictor}/{video_dir}_reprojected') / f'{video_name}_{cam}'
                    os.makedirs(output_cam_dir, exist_ok=True)

                    new_df = template_df.copy()
                    keypoints = new_df.columns.get_level_values('bodyparts').unique()
                    
                    for i, kp in enumerate(keypoints):
                        kp_idx = dataset_params['keypoints'].index(kp)
                        kp_x_cols = new_df.columns[
                            (new_df.columns.get_level_values('bodyparts') == kp) & 
                            (new_df.columns.get_level_values('coords') == 'x')
                        ]
                        kp_y_cols = new_df.columns[
                            (new_df.columns.get_level_values('bodyparts') == kp) & 
                            (new_df.columns.get_level_values('coords') == 'y')
                        ]
                        
                        for col in kp_x_cols:
                            new_df[col] = points_2d[:, kp_idx, 0]
                        for col in kp_y_cols:
                            new_df[col] = points_2d[:, kp_idx, 1]
                    
                    # Save the reprojected points
                    snippet_output_file = output_cam_dir / snippet_file
                    new_df.to_csv(snippet_output_file)
                    print(f"Saved reprojected 2D points to {snippet_output_file}")

                    
                    # Construct bbox path in data directory
                    # Extract the original video name (without camera suffix) and camera from the output path
                    output_path_parts = Path(snippet_output_file).parts
                    full_video_name = output_path_parts[-2]  # e.g., "PRL43_200617_131904_lBack"
                    snippet_name = output_path_parts[-1]  # e.g., "img00000005_uncropped.csv"

                    # For chickadee-crop, the video_name already includes the camera
                    # So we use it directly
                    bbox_path = Path(data_dir) / "videos-for-each-labeled-frame" / full_video_name / snippet_name.replace("_uncropped.csv", "_bbox.csv")

                    if bbox_path.exists():
                        # Create cropped version
                        cropped_file = snippet_output_file.with_stem(snippet_output_file.stem.replace('_uncropped', ''))
                        generate_cropped_csv_file(
                            input_csv_file=str(snippet_output_file),
                            input_bbox_file=str(bbox_path),
                            output_csv_file=str(cropped_file),
                            img_height=320,
                            img_width=320,
                            mode="subtract"
                        )
                        print(f"Created cropped file: {cropped_file}")
                    else:
                        print(f"No bbox file found at {bbox_path}, skipping cropped file creation")

creating anipose predictions for mvt_3d_loss_3110_0
video: CSDS-Day1-A_1-Defeat_
calibration file: /teamspace/studios/data/two-mouse/calibrations/CSDS-Day1-A_1-Defeat.toml
checking single file path: /teamspace/studios/this_studio/outputs/two-mouse/test_3110_MVT_distil_patch_masking/mvt_3d_loss_3110_0/videos_new/CSDS-Day1-A_1-Defeat_Camera0.csv
checking single file path: /teamspace/studios/this_studio/outputs/two-mouse/test_3110_MVT_distil_patch_masking/mvt_3d_loss_3110_0/videos_new/CSDS-Day1-A_1-Defeat_Camera1.csv
checking single file path: /teamspace/studios/this_studio/outputs/two-mouse/test_3110_MVT_distil_patch_masking/mvt_3d_loss_3110_0/videos_new/CSDS-Day1-A_1-Defeat_Camera2.csv
checking single file path: /teamspace/studios/this_studio/outputs/two-mouse/test_3110_MVT_distil_patch_masking/mvt_3d_loss_3110_0/videos_new/CSDS-Day1-A_1-Defeat_Camera3.csv
checking single file path: /teamspace/studios/this_studio/outputs/two-mouse/test_3110_MVT_distil_patch_masking/mvt_3d_loss_3110_0/vi

In [ ]:
# # Run inference on all InD/OOD videos and compute unsupervised metrics
# cfg_pipe , cfg_lp = load_cfgs('/teamspace/studios/this_studio/lp3d-analysis/configs/pipeline_inference.yaml')
# video_dir = 'videos-for-each-labeled-frame_reprojected'
# overwrite = False

# if dataset_name == 'fly-anipose':
#     new_csv_files = [f for f in os.listdir(data_dir) if f.endswith('_new.csv')]
# elif dataset_name == 'chickadee-crop':    
#     # I need it to also have the word remapped in the name
#     # the below is for the uncropped csv files
#     # new_csv_files = [f for f in os.listdir(data_dir) if f.endswith('_new.csv') and 'remapped' in f]
    
#     # I want the files that don't have remapped in the name
#     new_csv_files = [f for f in os.listdir(data_dir) if f.endswith('_new.csv') and 'remapped' not in f]

# print(f"the new csv files are {new_csv_files}")

# # Process each predictor
# for predictor in predictors:
#     print(f'Processing predictor: {predictor}')
    
#     # Use dynamic results directory based on predictor
#     results_dir_member = f'/teamspace/studios/this_studio/outputs/{dataset_name}/{predictor}'
    
#     for csv_file in new_csv_files:
#         # load original csv 
#         file_path = os.path.join(data_dir, csv_file)
#         print(f"the file path is {file_path}")
#         original_df = pd.read_csv(file_path, header=[0,1,2], index_col=0)
#         original_df = io_utils.fix_empty_first_row(original_df) 
#         original_index = original_df.index  
#         # Debug the first frame in the index
#         print(f"Original index has {len(original_index)} entries")
#         if len(original_index) > 0:
#             first_frame = original_index[0]
#             print(f"First frame in index: {first_frame}")
        
#         # load each of the new csv files and iterate through the index 
#         prediction_name = '_'.join(csv_file.split('_')[1:])
#         print(f"the prediction name is {prediction_name}")
#         preds_file = os.path.join(results_dir_member, video_dir, f'predictions_{prediction_name}') 
#         print(f"the preds file is {preds_file}")
#         if os.path.exists(preds_file) and not overwrite:
#             print(f'Predictions file {preds_file} already exists. Skipping.')
#             continue
#         results_list = []
        
#         # for img_path in df.index:
#         for img_path in original_index:
#             # print(f" the original index is {original_index}")
#             # print(f"the img path is {img_path}")
#             # process the paths 
#             relative_img_path = '/'.join(img_path.split('/')[1:]) # removed 'labeled-data/'
#             # add _uncropped in the end of the path before the mp4
#             # print(f"the relative img path is {relative_img_path}") 
#             # if dataset_name == 'chickadee-crop':
#             #     relative_img_path = relative_img_path.replace('.png', '_uncropped.png')
#             #     print(f"the relative img path is {relative_img_path}")

#             snippet_path = relative_img_path.replace('png', 'mp4')
#             # print(f"the snippet path is {snippet_path}")
#             # Load the 51-frame csv file 
#             snippet_file = os.path.join(results_dir_member, video_dir , snippet_path.replace('mp4', 'csv'))
#             # print(f"the snippet file is {snippet_file}")

#             if os.path.exists(snippet_file):
#                 snippet_df = pd.read_csv(snippet_file, header=[0,1,2], index_col=0)
#                 snippet_df = io_utils.fix_empty_first_row(snippet_df)

#                 # extract center frame 
#                 assert snippet_df.shape[0] % 2 != 0 # ensure odd number of frames
#                 idx_frame = int(np.floor(snippet_df.shape[0] / 2))            
#                 # create results with original image path as index 
#                 result = snippet_df[snippet_df.index == idx_frame].rename(index={idx_frame: img_path})
#                 results_list.append(result)

#         # combine all results 
#         if results_list:
#             results_df = pd.concat(results_list)
#             # results_df.sort_index(inplace=True) # this is maybe for the chickadee
#             results_df = results_df.reindex(original_index)
            
#             # Add "set" column so this df is interpreted as labeled data predictions
#             results_df.loc[:,("set", "", "")] = "train"
#             results_df.to_csv(preds_file)
#             print(f'Saved predictions to {preds_file}')

#             try:
#                 compute_metrics_single(
#                     cfg=cfg_lp,
#                     labels_file=os.path.join(cfg_lp.data.data_dir, csv_file),
#                     preds_file=preds_file,
#                     data_module=None
#                 )
#                 print(f"Succesfully computed metrics for {preds_file}")
#             except Exception as e:
#                 print(f"Error computing metrics\n{e}")
#                 print(traceback.format_exc())

the new csv files are ['CollectedData_Cam-A_new.csv', 'CollectedData_Cam-B_new.csv', 'CollectedData_Cam-C_new.csv', 'CollectedData_Cam-D_new.csv', 'CollectedData_Cam-E_new.csv', 'CollectedData_Cam-F_new.csv', 'calibrations_new.csv']
Processing predictor: supervised_400_1
the file path is /teamspace/studios/data/fly-anipose/CollectedData_Cam-A_new.csv
Original index has 300 entries
First frame in index: labeled-data/05272019_fly4_0_R2C26_Cam-A_rot-ccw-0.18_sec/img00000014.png
the prediction name is Cam-A_new.csv
the preds file is /teamspace/studios/this_studio/outputs/fly-anipose/supervised_400_1/videos-for-each-labeled-frame_reprojected/predictions_Cam-A_new.csv
the file path is /teamspace/studios/data/fly-anipose/CollectedData_Cam-B_new.csv
Original index has 300 entries
First frame in index: labeled-data/05272019_fly4_0_R2C26_Cam-B_rot-ccw-0.18_sec/img00000014.png
the prediction name is Cam-B_new.csv
the preds file is /teamspace/studios/this_studio/outputs/fly-anipose/supervised_400_

# making pixel error files for a single network 

In [6]:
# Run inference on all InD/OOD videos and compute unsupervised metrics
cfg_pipe , cfg_lp = load_cfgs('/teamspace/studios/this_studio/lp3d-analysis/configs/pipeline_inference.yaml')
# video_dir_original = 'videos-for-each-labeled-frame'
video_dir = 'videos-for-each-labeled-frame_reprojected'
# model_dir = '/teamspace/studios/this_studio/outputs/fly-anipose/results_100_5k/supervised_100_0'
results_dir_member = f'/teamspace/studios/this_studio/outputs/{dataset_name}/test_200_MVT_3d_loss_patch_masking/mvt_3d_loss_200_0-2/ensemble_median'
# results_dir_member = f'/teamspace/studios/this_studio/outputs/{dataset_name}/test_3218_MVT_distil_patch_masking_non_lin_varinf/mvt_3d_loss_3218_0'

overwrite = False

if dataset_name == 'fly-anipose':
    new_csv_files = [f for f in os.listdir(data_dir) if f.endswith('_new.csv')]
elif dataset_name == 'chickadee-crop':    
    # I need it to also have the word remapped in the name
    new_csv_files = [f for f in os.listdir(data_dir) if f.endswith('_new.csv') and not 'remapped' in f]

print(f"the new csv files are {new_csv_files}")

for csv_file in new_csv_files:
    # load original csv 
    file_path = os.path.join(data_dir, csv_file)
    print(f"the file path is {file_path}")
    original_df = pd.read_csv(file_path, header=[0,1,2], index_col=0)
    original_df = io_utils.fix_empty_first_row(original_df) 
    original_index = original_df.index  
    
    # Debug the first frame in the index
    print(f"Original index has {len(original_index)} entries")
    if len(original_index) > 0:
        first_frame = original_index[0]
        print(f"First frame in index: {first_frame}")

    # load each of the new csv files and iterate through the index 
    prediction_name = '_'.join(csv_file.split('_')[1:])
    preds_file = os.path.join(results_dir_member, video_dir, f'predictions_{prediction_name}') 
    #print(f"the preds file is {preds_file}")
    if os.path.exists(preds_file) and not overwrite:
        print(f'Predictions file {preds_file} already exists. Skipping.')
        continue
    
    results_list = []
    #file_path = os.path.join(data_dir, csv_file)
    #df = pd.read_csv(file_path, header=[0,1,2], index_col=0)
    
    # for img_path in df.index:
    for img_path in original_index:
        # process the paths 
        relative_img_path = '/'.join(img_path.split('/')[1:]) # removed 'labeled-data/'
        snippet_path = relative_img_path.replace('png', 'mp4')
        # Load the 51-frame csv file 
        snippet_file = os.path.join(results_dir_member, video_dir , snippet_path.replace('mp4', 'csv'))

        if os.path.exists(snippet_file):
            snippet_df = pd.read_csv(snippet_file, header=[0,1,2], index_col=0)
            snippet_df = io_utils.fix_empty_first_row(snippet_df)

            # extract center frame 
            assert snippet_df.shape[0] % 2 != 0 # ensure odd number of frames
            idx_frame = int(np.floor(snippet_df.shape[0] / 2))
            
            # create results with original image path as index 
            result = snippet_df[snippet_df.index == idx_frame].rename(index={idx_frame: img_path})
            results_list.append(result)

    # combine all results 
    if results_list:
        results_df = pd.concat(results_list)
        #results_df.sort_index(inplace=True)
        results_df = results_df.reindex(original_index)

        # Add "set" column so this df is interpreted as labeled data predictions
        results_df.loc[:,("set", "", "")] = "train"
        results_df.to_csv(preds_file)
        print(f'Saved predictions to {preds_file}')

    
        try:
            compute_metrics_single(
                cfg=cfg_lp,
                labels_file=os.path.join(cfg_lp.data.data_dir, csv_file),
                preds_file=preds_file,
                data_module=None
            )
            print(f"Succesfully computed metrics for {preds_file}")
        except Exception as e:
            print(f"Error computing metrics\n{e}")
            print(traceback.format_exc())




the new csv files are ['CollectedData_lBack_new.csv', 'CollectedData_lFront_new.csv', 'CollectedData_lTop_new.csv', 'CollectedData_rBack_new.csv', 'CollectedData_rFront_new.csv', 'CollectedData_rTop_new.csv', 'bboxes_lBack_new.csv', 'bboxes_lFront_new.csv', 'bboxes_lTop_new.csv', 'bboxes_rBack_new.csv', 'bboxes_rFront_new.csv', 'bboxes_rTop_new.csv', 'calibrations_new.csv']
the file path is /teamspace/studios/data/chickadee-crop/CollectedData_lBack_new.csv
Original index has 143 entries
First frame in index: labeled-data/PRL43_200617_131904_lBack/img00000005.png
Saved predictions to /teamspace/studios/this_studio/outputs/chickadee-crop/test_200_MVT_3d_loss_patch_masking/mvt_3d_loss_200_0-2/ensemble_median/videos-for-each-labeled-frame_reprojected/predictions_lBack_new.csv
Succesfully computed metrics for /teamspace/studios/this_studio/outputs/chickadee-crop/test_200_MVT_3d_loss_patch_masking/mvt_3d_loss_200_0-2/ensemble_median/videos-for-each-labeled-frame_reprojected/predictions_lBack

# post process all networks (ensemble methods)

In [ ]:
# # results_dir2 = Path(f'{results_dir}/results_100_5k/ensemble_median/{video_dir}_reprojected')
# base_dir = Path(f'{results_dir}/results_100_5k')
# # print(results_dir2)
# model_type = 'supervised'
# seed_range = [0,4]
# n_labels = 100
# mode = 'ensemble_median'
# views = cam_names
# cfg_pipe , cfg_lp = load_cfgs('/teamspace/studios/this_studio/lp3d-analysis/configs/pipeline_inference.yaml')


# print(base_dir)
# ensemble_dir, seed_dirs, _ = setup_ensemble_dirs(
#     base_dir, model_type, n_labels, seed_range, mode, ""
# )

# print(ensemble_dir)
# print(seed_dirs)

# inference_dir_original = f'{video_dir}'
# inference_dir = f'{video_dir}_reprojected'

# first_seed_dir = seed_dirs[0]
# inf_dir_in_first_seed = os.path.join(first_seed_dir, inference_dir_original)
# print(first_seed_dir)

# entries = os.listdir(inf_dir_in_first_seed)
# has_subdirectories = any(os.path.isdir(os.path.join(inf_dir_in_first_seed, entry)) for entry in entries)
# print(has_subdirectories)

# # Get original directory structure
# original_structure = get_original_structure(first_seed_dir, inference_dir_original, views)

# output_dir = os.path.join(ensemble_dir, mode, inference_dir)
# os.makedirs(output_dir, exist_ok=True)

# for view in views:
#     print(f"\nProcessing view: {view}")
    
#     view_dirs = {
#         dir_name: files 
#         for dir_name, files in original_structure.items()
#         if view in dir_name
#     }
    
#     # Process each view-specific directory
#     for original_dir, csv_files in view_dirs.items():
#         process_singleview_directory(
#             original_dir, csv_files, view, 
#             seed_dirs, inference_dir, output_dir, mode
#         )
    
#     # Process final predictions for this view
#     process_final_predictions(
#         view=view,
#         output_dir=output_dir,
#         seed_dirs=seed_dirs,
#         inference_dir=inference_dir_original,
#         cfg_lp=cfg_lp
#     )

/teamspace/studios/this_studio/outputs/chickadee-crop/results_100_5k
/teamspace/studios/this_studio/outputs/chickadee-crop/results_100_5k/supervised_100_0-4
['/teamspace/studios/this_studio/outputs/chickadee-crop/results_100_5k/supervised_100_0', '/teamspace/studios/this_studio/outputs/chickadee-crop/results_100_5k/supervised_100_1', '/teamspace/studios/this_studio/outputs/chickadee-crop/results_100_5k/supervised_100_2', '/teamspace/studios/this_studio/outputs/chickadee-crop/results_100_5k/supervised_100_3', '/teamspace/studios/this_studio/outputs/chickadee-crop/results_100_5k/supervised_100_4']
/teamspace/studios/this_studio/outputs/chickadee-crop/results_100_5k/supervised_100_0
True
View lBack dirs: ['PRL43_200617_131904_lBack', 'PRL43_200701_142147_lBack', 'SLV151_200728_132004_lBack', 'SLV151_200730_131948_lBack', 'predictions_CollectedData_lBack_new.csv', 'predictions_CollectedData_lBack_new_pixel_error.csv', 'predictions_lBack_new.csv', 'predictions_lBack_new_pixel_error.csv']
Ch

# Trying to triangulate using 4 views only


In [4]:
# At the top, after cam_names = dataset_params['cam_names']
# Define which cameras to use for triangulation
triangulation_cameras = ['lBack', 'lFront', 'rBack', 'rFront']  # Your 4 reliable cameras
# triangulation_cameras = cam_names  # Use all cameras (current behavior)

# Get indices of triangulation cameras
triangulation_cam_indices = [cam_names.index(cam) for cam in triangulation_cameras]

In [ ]:
# results_df = []
# for predictor in predictors:
#     print(f"creating anipose predictions for {predictor}")
#     for video_name in video_names:
#         print(f'video: {video_name}')
#         calibration_file = Path(data_dir) / 'calibrations' / f'{video_name.replace("__", "_")}.toml' 
#         if '.short' in calibration_file.name:
#             calibration_file = calibration_file.with_name(calibration_file.name.replace('.short', ''))
#         # if there is .short in the calibration file, remove it
#         print(f'calibration file: {calibration_file}')
#         assert calibration_file.is_file()
        
#         # Check which directory structure we're dealing with
#         # First, try the single file structure
#         cam_files = {}
#         all_files_exist = True
#         single_file_structure = True
        
#         for cam in cam_names:
#             if dataset_name == 'fly-anipose':
#                 stem = video_name.replace("__", f"_{cam}_")
#                 data_path = Path(f'{results_dir}/{predictor}/{video_dir}') / f'{stem}.csv'
#             elif dataset_name == 'chickadee-crop':
#                 # stem = f'{video_name}_{cam}'
#                 # data_path = Path(f'{results_dir}/{predictor}/{video_dir}') / f'{stem}.csv'
#                 stem = f'{video_name}_{cam}.short_uncropped'
#                 data_path = Path(f'{results_dir}/{predictor}/{video_dir}') / f'{stem}.csv'
#             else:
#                 print(f"Unsupported dataset_name: {dataset_name}")
#                 all_files_exist = False
#                 break

#             print(f'checking single file path: {data_path}')
#             if data_path.is_file():
#                 cam_files[cam] = data_path
#             else:
#                 all_files_exist = False
#                 break
        
#         # If single file structure doesn't work, try snippet structure
#         if not all_files_exist:
#             print("Single file structure not found, trying snippet structure...")
#             single_file_structure = False
#             cam_dirs = {}
#             all_dirs_exist = True
            
#             for cam in cam_names:
#                 if dataset_name == 'fly-anipose':
#                     data_path = Path(f'{results_dir}/{predictor}/{video_dir}') / f'{video_name.replace("__", f"_{cam}_")}'
#                 elif dataset_name == 'chickadee-crop':
#                     data_path = Path(f'{results_dir}/{predictor}/{video_dir}') / f'{video_name}_{cam}'
#                     print(f'data_path: {data_path}')
                
#                 # print(f'checking snippet directory path: {data_path}')
#                 if data_path.is_dir():
#                     cam_dirs[cam] = data_path
#                 else:
#                     all_dirs_exist = False
#                     break
            
#             if not all_dirs_exist:
#                 print(f"Skipping video {video_name} - neither structure found")
#                 continue
#         else:
#             print("Using single file structure")

#         if single_file_structure:
#             # Process single file structure
#             snippet_files = [cam_files[cam_names[0]].name]
#             print(f" snippet_files: {snippet_files}")
            
#             for snippet_file in snippet_files:
#                 snippet_base = os.path.splitext(snippet_file)[0]
#                 print(f"Processing snippet: {snippet_base}")

#                 snippet_arrays = []
#                 all_snippets_valid = True
#                 template_df = None

#                 for cam in cam_names:
#                     cam_snippet_path = cam_files[cam]
#                     if not cam_snippet_path.exists():
#                         print(f"Warning: Snippet file {cam_snippet_path} does not exist. Skipping.")
#                         all_snippets_valid = False
#                         break

#                     try:
#                         snippet_df = pd.read_csv(cam_snippet_path, index_col=0, header=[0, 1, 2])
#                         if template_df is None:
#                             template_df = snippet_df.copy()
#                         xs = snippet_df.loc[:, snippet_df.columns.get_level_values('coords').isin(['x'])]
#                         ys = snippet_df.loc[:, snippet_df.columns.get_level_values('coords').isin(['y'])]
#                         snippet_arrays.append(np.stack([xs, ys], axis=2))
#                     except Exception as e:
#                         print(f"Error reading snippet file {cam_snippet_path}: {e}")
#                         all_snippets_valid = False
#                         break

#                 if not all_snippets_valid:
#                     print(f"Skipping snippet {snippet_file}")
#                     continue

#                 # Convert to numpy array and triangulate
#                 snippet_arrays = np.stack(snippet_arrays, axis=0)
#                 print(f'Snippet arrays shape: {snippet_arrays.shape}')

#                 # Filter to only triangulation cameras
#                 triangulation_arrays = snippet_arrays[triangulation_cam_indices]
#                 print(f'Triangulation arrays shape: {triangulation_arrays.shape}')

#                 camgroup = CameraGroup.load(calibration_file)
#                 print("Available methods:", [method for method in dir(camgroup) if 'triangulate' in method.lower()])

#                 # Create a subset camgroup with only triangulation cameras
#                 triangulation_camgroup = CameraGroup(
#                     cameras=[camgroup.cameras[i] for i in triangulation_cam_indices],
#                     metadata=camgroup.metadata
#                 )

#                 snippet_points_3d = []
#                 for k in range(n_keypoints):
#                     # print(f'triangulating keypoint {k}')
#                     # snippet_points_3d.append(camgroup.triangulate(snippet_arrays[:, :, k, :], fast=True))
#                     snippet_points_3d.append(triangulation_camgroup.triangulate(triangulation_arrays[:, :, k, :], fast=True))
#                 snippet_points_3d = np.stack(snippet_points_3d, axis=1)
#                 print(f'Snippet points_3d shape: {snippet_points_3d.shape}')

#                 for cam_idx, cam in enumerate(cam_names):
#                     camera = camgroup.cameras[cam_idx]

#                     points_flat = snippet_points_3d.reshape(-1, 3)
#                     points_2d_flat = camera.project(points_flat)
#                     n_frames = snippet_points_3d.shape[0]
#                     points_2d = points_2d_flat.reshape(n_frames, n_keypoints, 2)

#                     # this is for reprojection 
#                     # Calculate reprojection error
#                     original_2d = snippet_arrays[cam_idx]  # Original 2D points for this camera
#                     reprojection_error = np.sqrt(np.sum((original_2d - points_2d)**2, axis=2))  # Euclidean distance per keypoint per frame
#                     mean_error = np.mean(reprojection_error)
#                     median_error = np.median(reprojection_error)
#                     max_error = np.max(reprojection_error)
#                     std_error = np.std(reprojection_error)

#                     print(f"Camera {cam} reprojection error statistics:")
#                     print(f"  Mean error: {mean_error:.4f} pixels")
#                     print(f"  Median error: {median_error:.4f} pixels")
#                     print(f"  Max error: {max_error:.4f} pixels")
#                     print(f"  Std error: {std_error:.4f} pixels")
                    
#                     # Optional: Print per-keypoint errors
#                     keypoint_errors = np.mean(reprojection_error, axis=0)  # Average error per keypoint
#                     print(f"  Per-keypoint mean errors: {keypoint_errors}")
                    
#                     # Optional: Print per-frame errors
#                     frame_errors = np.mean(reprojection_error, axis=1)  # Average error per frame
#                     print(f"  Per-frame mean errors (first 10): {frame_errors[:10]}")

#                     # Create ONLY the main reprojected directory (no camera subdirectories)
#                     output_main_dir = Path(f'{results_dir}/{predictor}/{video_dir}_reprojected')
#                     os.makedirs(output_main_dir, exist_ok=True)
#                     print(f"Created main output directory: {output_main_dir}")

#                     new_df = template_df.copy()

#                     keypoints = new_df.columns.get_level_values('bodyparts').unique()
#                     for i, kp in enumerate(keypoints):
#                         kp_idx = dataset_params['keypoints'].index(kp)
#                         kp_x_cols = new_df.columns[
#                             (new_df.columns.get_level_values('bodyparts') == kp) &
#                             (new_df.columns.get_level_values('coords') == 'x')
#                         ]
#                         kp_y_cols = new_df.columns[
#                             (new_df.columns.get_level_values('bodyparts') == kp) &
#                             (new_df.columns.get_level_values('coords') == 'y')
#                         ]
#                         for col in kp_x_cols:
#                             new_df[col] = points_2d[:, kp_idx, 0]
#                         for col in kp_y_cols:
#                             new_df[col] = points_2d[:, kp_idx, 1]

#                     # Save CSV file directly in main reprojected directory
#                     out_name = cam_files[cam].name
#                     snippet_output_file = output_main_dir / out_name
#                     new_df.to_csv(snippet_output_file)
#                     print(f"Saved reprojected 2D points to {snippet_output_file}")

#                     # Construct bbox path for full videos
#                     # For chickadee-crop full videos, bbox files are in videos_new_fix directory
#                     # Pattern: {video_name}_{cam}.short_bbox.csv
#                     bbox_path = Path(data_dir) / "videos_new_fix" / f"{video_name}_{cam}.short_bbox.csv"

#                     if bbox_path.exists():
#                         # Create cropped version
#                         cropped_file = snippet_output_file.with_stem(snippet_output_file.stem.replace("_uncropped", ""))
#                         generate_cropped_csv_file(
#                             input_csv_file=str(snippet_output_file),
#                             input_bbox_file=str(bbox_path),
#                             output_csv_file=str(cropped_file),
#                             img_height=320,
#                             img_width=320,
#                             mode="subtract"
#                         )
#                         print(f"Created cropped file: {cropped_file}")
#                     else:
#                         print(f"No bbox file found at {bbox_path}, skipping cropped file creation")

                    

#         else:
#             # Process snippet structure
#             first_cam = cam_names[0]
#             if dataset_name == 'fly-anipose':
#                 snippet_files = sorted([f for f in os.listdir(cam_dirs[first_cam]) if f.endswith('.csv')])
#             elif dataset_name == 'chickadee-crop':
#                 snippet_files = sorted([f for f in os.listdir(cam_dirs[first_cam]) 
#                                     if f.endswith('uncropped.csv')])

#             # process each snippet individually
#             for snippet_file in snippet_files:
#                 snippet_base = os.path.splitext(snippet_file)[0]
#                 print(f"Processing snippet: {snippet_base}")
                
#                 snippet_arrays = []
#                 all_snippets_valid = True
#                 template_df = None

#                 for cam in cam_names:
#                     cam_snippet_path = cam_dirs[cam] / snippet_file
#                     if not cam_snippet_path.exists():
#                         print(f"Warning: Snippet file {cam_snippet_path} does not exist. Skipping.")
#                         all_snippets_valid = False
#                         break
                    
#                     try:
#                         snippet_df = pd.read_csv(cam_snippet_path, index_col=0, header=[0, 1, 2])
#                         if template_df is None:
#                             template_df = snippet_df.copy()
#                         xs = snippet_df.loc[:, snippet_df.columns.get_level_values('coords').isin(['x'])]
#                         ys = snippet_df.loc[:, snippet_df.columns.get_level_values('coords').isin(['y'])]
#                         snippet_arrays.append(np.stack([xs, ys], axis=2))
#                     except Exception as e:
#                         print(f"Error reading snippet file {cam_snippet_path}: {e}")
#                         all_snippets_valid = False
#                         break
                
#                 if not all_snippets_valid:
#                     print(f"Skipping snippet {snippet_file}")
#                     continue

#                 # Convert to numpy array and triangulate
#                 snippet_arrays = np.stack(snippet_arrays, axis=0)
#                 print(f'Snippet arrays shape: {snippet_arrays.shape}')
                
                
#                 camgroup = CameraGroup.load(calibration_file)
#                 snippet_points_3d = []
#                 for k in range(n_keypoints):
#                     snippet_points_3d.append(camgroup.triangulate(snippet_arrays[:, :, k, :], fast=True))
                
#                 snippet_points_3d = np.stack(snippet_points_3d, axis=1)
#                 print(f'Snippet points_3d shape: {snippet_points_3d.shape}')

#                 for cam_idx, cam in enumerate(cam_names):
#                     camera = camgroup.cameras[cam_idx]
#                     points_flat = snippet_points_3d.reshape(-1, 3)
#                     points_2d_flat = camera.project(points_flat)
#                     n_frames = snippet_points_3d.shape[0]
#                     points_2d = points_2d_flat.reshape(n_frames, n_keypoints, 2)

#                     # Create output directory for this camera
#                     if dataset_name == 'fly-anipose':
#                         output_cam_dir = Path(f'{results_dir}/{predictor}/{video_dir}_reprojected') / f'{video_name.replace("__", f"_{cam}_")}'
#                     elif dataset_name == 'chickadee-crop':
#                         output_cam_dir = Path(f'{results_dir}/{predictor}/{video_dir}_reprojected') / f'{video_name}_{cam}'
#                     os.makedirs(output_cam_dir, exist_ok=True)

#                     new_df = template_df.copy()
#                     keypoints = new_df.columns.get_level_values('bodyparts').unique()
                    
#                     for i, kp in enumerate(keypoints):
#                         kp_idx = dataset_params['keypoints'].index(kp)
#                         kp_x_cols = new_df.columns[
#                             (new_df.columns.get_level_values('bodyparts') == kp) & 
#                             (new_df.columns.get_level_values('coords') == 'x')
#                         ]
#                         kp_y_cols = new_df.columns[
#                             (new_df.columns.get_level_values('bodyparts') == kp) & 
#                             (new_df.columns.get_level_values('coords') == 'y')
#                         ]
                        
#                         for col in kp_x_cols:
#                             new_df[col] = points_2d[:, kp_idx, 0]
#                         for col in kp_y_cols:
#                             new_df[col] = points_2d[:, kp_idx, 1]
                    
#                     # Save the reprojected points
#                     snippet_output_file = output_cam_dir / snippet_file
#                     new_df.to_csv(snippet_output_file)
#                     print(f"Saved reprojected 2D points to {snippet_output_file}")

                    
#                     # Construct bbox path in data directory
#                     # Extract the original video name (without camera suffix) and camera from the output path
#                     output_path_parts = Path(snippet_output_file).parts
#                     full_video_name = output_path_parts[-2]  # e.g., "PRL43_200617_131904_lBack"
#                     snippet_name = output_path_parts[-1]  # e.g., "img00000005_uncropped.csv"

#                     # For chickadee-crop, the video_name already includes the camera
#                     # So we use it directly
#                     bbox_path = Path(data_dir) / "videos-for-each-labeled-frame" / full_video_name / snippet_name.replace("_uncropped.csv", "_bbox.csv")

#                     if bbox_path.exists():
#                         # Create cropped version
#                         cropped_file = snippet_output_file.with_stem(snippet_output_file.stem.replace('_uncropped', ''))
#                         generate_cropped_csv_file(
#                             input_csv_file=str(snippet_output_file),
#                             input_bbox_file=str(bbox_path),
#                             output_csv_file=str(cropped_file),
#                             img_height=320,
#                             img_width=320,
#                             mode="subtract"
#                         )
#                         print(f"Created cropped file: {cropped_file}")
#                     else:
#                         print(f"No bbox file found at {bbox_path}, skipping cropped file creation")

creating anipose predictions for mvt_3d_loss_200_0-2/ensemble_median
video: PRL43_200617_131904
calibration file: /teamspace/studios/data/chickadee-crop/calibrations/PRL43_200617_131904.toml
checking single file path: /teamspace/studios/this_studio/outputs/chickadee-crop/test_200_MVT_3d_loss_patch_masking/mvt_3d_loss_200_0-2/ensemble_median/videos-for-each-labeled-frame_reprojected/PRL43_200617_131904_lBack.short_uncropped.csv
Single file structure not found, trying snippet structure...
data_path: /teamspace/studios/this_studio/outputs/chickadee-crop/test_200_MVT_3d_loss_patch_masking/mvt_3d_loss_200_0-2/ensemble_median/videos-for-each-labeled-frame_reprojected/PRL43_200617_131904_lBack
Skipping video PRL43_200617_131904 - neither structure found
video: PRL43_200701_142147
calibration file: /teamspace/studios/data/chickadee-crop/calibrations/PRL43_200701_142147.toml
checking single file path: /teamspace/studios/this_studio/outputs/chickadee-crop/test_200_MVT_3d_loss_patch_masking/mvt_3

In [5]:
results_df = []
for predictor in predictors:
    print(f"creating anipose predictions for {predictor}")
    for video_name in video_names:
        print(f'video: {video_name}')
        calibration_file = Path(data_dir) / 'calibrations' / f'{video_name.replace("__", "_")}.toml' 
        if '.short' in calibration_file.name:
            calibration_file = calibration_file.with_name(calibration_file.name.replace('.short', ''))
        # if there is .short in the calibration file, remove it
        print(f'calibration file: {calibration_file}')
        assert calibration_file.is_file()
        
        # Check which directory structure we're dealing with
        # First, try the single file structure
        cam_files = {}
        all_files_exist = True
        single_file_structure = True
        
        for cam in cam_names:
            if dataset_name == 'fly-anipose':
                stem = video_name.replace("__", f"_{cam}_")
                data_path = Path(f'{results_dir}/{predictor}/{video_dir}') / f'{stem}.csv'
            elif dataset_name == 'chickadee-crop':
                # stem = f'{video_name}_{cam}'
                # data_path = Path(f'{results_dir}/{predictor}/{video_dir}') / f'{stem}.csv'
                stem = f'{video_name}_{cam}.short_uncropped'
                data_path = Path(f'{results_dir}/{predictor}/{video_dir}') / f'{stem}.csv'
            else:
                print(f"Unsupported dataset_name: {dataset_name}")
                all_files_exist = False
                break

            print(f'checking single file path: {data_path}')
            if data_path.is_file():
                cam_files[cam] = data_path
            else:
                all_files_exist = False
                break
        
        # If single file structure doesn't work, try snippet structure
        if not all_files_exist:
            print("Single file structure not found, trying snippet structure...")
            single_file_structure = False
            cam_dirs = {}
            all_dirs_exist = True
            
            for cam in cam_names:
                if dataset_name == 'fly-anipose':
                    data_path = Path(f'{results_dir}/{predictor}/{video_dir}') / f'{video_name.replace("__", f"_{cam}_")}'
                elif dataset_name == 'chickadee-crop':
                    data_path = Path(f'{results_dir}/{predictor}/{video_dir}') / f'{video_name}_{cam}'
                    print(f'data_path: {data_path}')
                
                # print(f'checking snippet directory path: {data_path}')
                if data_path.is_dir():
                    cam_dirs[cam] = data_path
                else:
                    all_dirs_exist = False
                    break
            
            if not all_dirs_exist:
                print(f"Skipping video {video_name} - neither structure found")
                continue
        else:
            print("Using single file structure")

        if single_file_structure:
            # Process single file structure
            snippet_files = [cam_files[cam_names[0]].name]
            print(f" snippet_files: {snippet_files}")
            
            for snippet_file in snippet_files:
                snippet_base = os.path.splitext(snippet_file)[0]
                print(f"Processing snippet: {snippet_base}")

                snippet_arrays = []
                all_snippets_valid = True
                template_df = None

                for cam in cam_names:
                    cam_snippet_path = cam_files[cam]
                    if not cam_snippet_path.exists():
                        print(f"Warning: Snippet file {cam_snippet_path} does not exist. Skipping.")
                        all_snippets_valid = False
                        break

                    try:
                        snippet_df = pd.read_csv(cam_snippet_path, index_col=0, header=[0, 1, 2])
                        if template_df is None:
                            template_df = snippet_df.copy()
                        xs = snippet_df.loc[:, snippet_df.columns.get_level_values('coords').isin(['x'])]
                        ys = snippet_df.loc[:, snippet_df.columns.get_level_values('coords').isin(['y'])]
                        snippet_arrays.append(np.stack([xs, ys], axis=2))
                    except Exception as e:
                        print(f"Error reading snippet file {cam_snippet_path}: {e}")
                        all_snippets_valid = False
                        break

                if not all_snippets_valid:
                    print(f"Skipping snippet {snippet_file}")
                    continue

                # Convert to numpy array and triangulate
                snippet_arrays = np.stack(snippet_arrays, axis=0)
                print(f'Snippet arrays shape: {snippet_arrays.shape}')

                # Filter to only triangulation cameras
                triangulation_arrays = snippet_arrays[triangulation_cam_indices]
                print(f'Triangulation arrays shape: {triangulation_arrays.shape}')

                camgroup = CameraGroup.load(calibration_file)
                print("Available methods:", [method for method in dir(camgroup) if 'triangulate' in method.lower()])

                # Create a subset camgroup with only triangulation cameras
                triangulation_camgroup = CameraGroup(
                    cameras=[camgroup.cameras[i] for i in triangulation_cam_indices],
                    metadata=camgroup.metadata
                )

                snippet_points_3d = []
                for k in range(n_keypoints):
                    # print(f'triangulating keypoint {k}')
                    # snippet_points_3d.append(camgroup.triangulate(snippet_arrays[:, :, k, :], fast=True))
                    snippet_points_3d.append(triangulation_camgroup.triangulate(triangulation_arrays[:, :, k, :], fast=True))
                snippet_points_3d = np.stack(snippet_points_3d, axis=1)
                print(f'Snippet points_3d shape: {snippet_points_3d.shape}')

                for cam_idx, cam in enumerate(cam_names):
                    camera = camgroup.cameras[cam_idx]

                    points_flat = snippet_points_3d.reshape(-1, 3)
                    points_2d_flat = camera.project(points_flat)
                    n_frames = snippet_points_3d.shape[0]
                    points_2d = points_2d_flat.reshape(n_frames, n_keypoints, 2)

                    # this is for reprojection 
                    # Calculate reprojection error
                    original_2d = snippet_arrays[cam_idx]  # Original 2D points for this camera
                    reprojection_error = np.sqrt(np.sum((original_2d - points_2d)**2, axis=2))  # Euclidean distance per keypoint per frame
                    mean_error = np.mean(reprojection_error)
                    median_error = np.median(reprojection_error)
                    max_error = np.max(reprojection_error)
                    std_error = np.std(reprojection_error)

                    print(f"Camera {cam} reprojection error statistics:")
                    print(f"  Mean error: {mean_error:.4f} pixels")
                    print(f"  Median error: {median_error:.4f} pixels")
                    print(f"  Max error: {max_error:.4f} pixels")
                    print(f"  Std error: {std_error:.4f} pixels")
                    
                    # Optional: Print per-keypoint errors
                    keypoint_errors = np.mean(reprojection_error, axis=0)  # Average error per keypoint
                    print(f"  Per-keypoint mean errors: {keypoint_errors}")
                    
                    # Optional: Print per-frame errors
                    frame_errors = np.mean(reprojection_error, axis=1)  # Average error per frame
                    print(f"  Per-frame mean errors (first 10): {frame_errors[:10]}")

                    # Create ONLY the main reprojected directory (no camera subdirectories)
                    output_main_dir = Path(f'{results_dir}/{predictor}/{video_dir}_reprojected')
                    os.makedirs(output_main_dir, exist_ok=True)
                    print(f"Created main output directory: {output_main_dir}")

                    new_df = template_df.copy()

                    keypoints = new_df.columns.get_level_values('bodyparts').unique()
                    for i, kp in enumerate(keypoints):
                        kp_idx = dataset_params['keypoints'].index(kp)
                        kp_x_cols = new_df.columns[
                            (new_df.columns.get_level_values('bodyparts') == kp) &
                            (new_df.columns.get_level_values('coords') == 'x')
                        ]
                        kp_y_cols = new_df.columns[
                            (new_df.columns.get_level_values('bodyparts') == kp) &
                            (new_df.columns.get_level_values('coords') == 'y')
                        ]
                        for col in kp_x_cols:
                            new_df[col] = points_2d[:, kp_idx, 0]
                        for col in kp_y_cols:
                            new_df[col] = points_2d[:, kp_idx, 1]

                    # Save CSV file directly in main reprojected directory
                    out_name = cam_files[cam].name
                    snippet_output_file = output_main_dir / out_name
                    new_df.to_csv(snippet_output_file)
                    print(f"Saved reprojected 2D points to {snippet_output_file}")

                    # Construct bbox path for full videos
                    # For chickadee-crop full videos, bbox files are in videos_new_fix directory
                    # Pattern: {video_name}_{cam}.short_bbox.csv
                    bbox_path = Path(data_dir) / "videos_new_fix" / f"{video_name}_{cam}.short_bbox.csv"

                    if bbox_path.exists():
                        # Create cropped version
                        cropped_file = snippet_output_file.with_stem(snippet_output_file.stem.replace("_uncropped", ""))
                        generate_cropped_csv_file(
                            input_csv_file=str(snippet_output_file),
                            input_bbox_file=str(bbox_path),
                            output_csv_file=str(cropped_file),
                            img_height=320,
                            img_width=320,
                            mode="subtract"
                        )
                        print(f"Created cropped file: {cropped_file}")
                    else:
                        print(f"No bbox file found at {bbox_path}, skipping cropped file creation")

                    

        else:
            # Process snippet structure
            first_cam = cam_names[0]
            if dataset_name == 'fly-anipose':
                snippet_files = sorted([f for f in os.listdir(cam_dirs[first_cam]) if f.endswith('.csv')])
            elif dataset_name == 'chickadee-crop':
                snippet_files = sorted([f for f in os.listdir(cam_dirs[first_cam]) 
                                    if f.endswith('uncropped.csv')])

            # process each snippet individually
            for snippet_file in snippet_files:
                snippet_base = os.path.splitext(snippet_file)[0]
                print(f"Processing snippet: {snippet_base}")
                
                snippet_arrays = []
                all_snippets_valid = True
                template_df = None

                for cam in cam_names:
                    cam_snippet_path = cam_dirs[cam] / snippet_file
                    if not cam_snippet_path.exists():
                        print(f"Warning: Snippet file {cam_snippet_path} does not exist. Skipping.")
                        all_snippets_valid = False
                        break
                    
                    try:
                        snippet_df = pd.read_csv(cam_snippet_path, index_col=0, header=[0, 1, 2])
                        if template_df is None:
                            template_df = snippet_df.copy()
                        xs = snippet_df.loc[:, snippet_df.columns.get_level_values('coords').isin(['x'])]
                        ys = snippet_df.loc[:, snippet_df.columns.get_level_values('coords').isin(['y'])]
                        snippet_arrays.append(np.stack([xs, ys], axis=2))
                    except Exception as e:
                        print(f"Error reading snippet file {cam_snippet_path}: {e}")
                        all_snippets_valid = False
                        break
                
                if not all_snippets_valid:
                    print(f"Skipping snippet {snippet_file}")
                    continue

                # Convert to numpy array and triangulate
                snippet_arrays = np.stack(snippet_arrays, axis=0)
                print(f'Snippet arrays shape: {snippet_arrays.shape}')
                
                # Filter to only triangulation cameras
                triangulation_arrays = snippet_arrays[triangulation_cam_indices]
                print(f'Triangulation arrays shape: {triangulation_arrays.shape}')
                
                camgroup = CameraGroup.load(calibration_file)
                print("Available methods:", [method for method in dir(camgroup) if 'triangulate' in method.lower()])

                # Create a subset camgroup with only triangulation cameras
                triangulation_camgroup = CameraGroup(
                    cameras=[camgroup.cameras[i] for i in triangulation_cam_indices],
                    metadata=camgroup.metadata
                )

                snippet_points_3d = []
                for k in range(n_keypoints):
                    snippet_points_3d.append(triangulation_camgroup.triangulate(triangulation_arrays[:, :, k, :], fast=True))
                
                snippet_points_3d = np.stack(snippet_points_3d, axis=1)
                print(f'Snippet points_3d shape: {snippet_points_3d.shape}')

                for cam_idx, cam in enumerate(cam_names):
                    camera = camgroup.cameras[cam_idx]
                    points_flat = snippet_points_3d.reshape(-1, 3)
                    points_2d_flat = camera.project(points_flat)
                    n_frames = snippet_points_3d.shape[0]
                    points_2d = points_2d_flat.reshape(n_frames, n_keypoints, 2)

                    # Create output directory for this camera
                    if dataset_name == 'fly-anipose':
                        output_cam_dir = Path(f'{results_dir}/{predictor}/{video_dir}_reprojected') / f'{video_name.replace("__", f"_{cam}_")}'
                    elif dataset_name == 'chickadee-crop':
                        output_cam_dir = Path(f'{results_dir}/{predictor}/{video_dir}_reprojected') / f'{video_name}_{cam}'
                    os.makedirs(output_cam_dir, exist_ok=True)

                    new_df = template_df.copy()
                    keypoints = new_df.columns.get_level_values('bodyparts').unique()
                    
                    for i, kp in enumerate(keypoints):
                        kp_idx = dataset_params['keypoints'].index(kp)
                        kp_x_cols = new_df.columns[
                            (new_df.columns.get_level_values('bodyparts') == kp) & 
                            (new_df.columns.get_level_values('coords') == 'x')
                        ]
                        kp_y_cols = new_df.columns[
                            (new_df.columns.get_level_values('bodyparts') == kp) & 
                            (new_df.columns.get_level_values('coords') == 'y')
                        ]
                        
                        for col in kp_x_cols:
                            new_df[col] = points_2d[:, kp_idx, 0]
                        for col in kp_y_cols:
                            new_df[col] = points_2d[:, kp_idx, 1]
                    
                    # Save the reprojected points
                    snippet_output_file = output_cam_dir / snippet_file
                    new_df.to_csv(snippet_output_file)
                    print(f"Saved reprojected 2D points to {snippet_output_file}")

                    
                    # Construct bbox path in data directory
                    # Extract the original video name (without camera suffix) and camera from the output path
                    output_path_parts = Path(snippet_output_file).parts
                    full_video_name = output_path_parts[-2]  # e.g., "PRL43_200617_131904_lBack"
                    snippet_name = output_path_parts[-1]  # e.g., "img00000005_uncropped.csv"

                    # For chickadee-crop, the video_name already includes the camera
                    # So we use it directly
                    bbox_path = Path(data_dir) / "videos-for-each-labeled-frame" / full_video_name / snippet_name.replace("_uncropped.csv", "_bbox.csv")

                    if bbox_path.exists():
                        # Create cropped version
                        cropped_file = snippet_output_file.with_stem(snippet_output_file.stem.replace('_uncropped', ''))
                        generate_cropped_csv_file(
                            input_csv_file=str(snippet_output_file),
                            input_bbox_file=str(bbox_path),
                            output_csv_file=str(cropped_file),
                            img_height=320,
                            img_width=320,
                            mode="subtract"
                        )
                        print(f"Created cropped file: {cropped_file}")
                    else:
                        print(f"No bbox file found at {bbox_path}, skipping cropped file creation")

creating anipose predictions for mvt_3d_loss_200_0-2/ensemble_median
video: PRL43_200617_131904
calibration file: /teamspace/studios/data/chickadee-crop/calibrations/PRL43_200617_131904.toml
checking single file path: /teamspace/studios/this_studio/outputs/chickadee-crop/test_200_MVT_3d_loss_patch_masking/mvt_3d_loss_200_0-2/ensemble_median/videos-for-each-labeled-frame/PRL43_200617_131904_lBack.short_uncropped.csv
Single file structure not found, trying snippet structure...
data_path: /teamspace/studios/this_studio/outputs/chickadee-crop/test_200_MVT_3d_loss_patch_masking/mvt_3d_loss_200_0-2/ensemble_median/videos-for-each-labeled-frame/PRL43_200617_131904_lBack
data_path: /teamspace/studios/this_studio/outputs/chickadee-crop/test_200_MVT_3d_loss_patch_masking/mvt_3d_loss_200_0-2/ensemble_median/videos-for-each-labeled-frame/PRL43_200617_131904_lFront
data_path: /teamspace/studios/this_studio/outputs/chickadee-crop/test_200_MVT_3d_loss_patch_masking/mvt_3d_loss_200_0-2/ensemble_median

In [6]:
import pandas as pd
import os
from pathlib import Path
from lp3d_analysis.utils import generate_cropped_csv_file  # Correct import

# Define paths
data_dir = '/teamspace/studios/data/chickadee-crop'
output_dir = '/teamspace/studios/this_studio/outputs/chickadee-crop/test_200_MVT_3d_loss_patch_masking/mvt_3d_loss_200_0-2/ensemble_median/videos-for-each-labeled-frame'

# Get all video directories (these have names like "PRL43_200617_131904_lBack")
video_dirs = [d for d in os.listdir(output_dir) if os.path.isdir(os.path.join(output_dir, d)) and not d.endswith('.csv')]

for video_dir in video_dirs:
    print(f"Processing video directory: {video_dir}")
    
    # Path to the output CSV directory
    csv_dir = Path(output_dir) / video_dir
    
    # Path to the corresponding data directory with bbox files
    data_bbox_dir = Path(data_dir) / "videos-for-each-labeled-frame" / video_dir
    
    if not data_bbox_dir.exists():
        print(f"Data bbox directory not found: {data_bbox_dir}")
        continue
    
    # Process each cropped CSV file in the output directory
    for csv_file in os.listdir(csv_dir):
        if not csv_file.endswith('.csv'):
            continue
            
        cropped_file = csv_dir / csv_file
        
        # The bbox file has the same base name but with "_bbox.csv" extension
        bbox_filename = csv_file.replace('.csv', '_bbox.csv')
        bbox_file = data_bbox_dir / bbox_filename
        
        if not bbox_file.exists():
            print(f"Bbox file not found: {bbox_file}")
            continue
        
        # Create uncropped filename by adding "_uncropped" before .csv
        uncropped_filename = csv_file.replace('.csv', '_uncropped.csv')
        uncropped_file = csv_dir / uncropped_filename
        
        print(f"Converting {csv_file} to {uncropped_filename}")
        
        try:
            # Use mode="add" to convert cropped coordinates back to uncropped
            generate_cropped_csv_file(
                input_csv_file=str(cropped_file),
                input_bbox_file=str(bbox_file),
                output_csv_file=str(uncropped_file),
                img_height=320,
                img_width=320,
                mode="add"  # This adds the bbox offsets back to get uncropped coordinates
            )
            print(f"Created uncropped file: {uncropped_file}")
        except Exception as e:
            print(f"Error processing {csv_file}: {e}")

print("Finished creating uncropped files!")

Processing video directory: PRL43_200617_131904_lBack
Converting img00000005.csv to img00000005_uncropped.csv
Created uncropped file: /teamspace/studios/this_studio/outputs/chickadee-crop/test_200_MVT_3d_loss_patch_masking/mvt_3d_loss_200_0-2/ensemble_median/videos-for-each-labeled-frame/PRL43_200617_131904_lBack/img00000005_uncropped.csv
Converting img00000010.csv to img00000010_uncropped.csv
Created uncropped file: /teamspace/studios/this_studio/outputs/chickadee-crop/test_200_MVT_3d_loss_patch_masking/mvt_3d_loss_200_0-2/ensemble_median/videos-for-each-labeled-frame/PRL43_200617_131904_lBack/img00000010_uncropped.csv
Converting img00030000.csv to img00030000_uncropped.csv
Created uncropped file: /teamspace/studios/this_studio/outputs/chickadee-crop/test_200_MVT_3d_loss_patch_masking/mvt_3d_loss_200_0-2/ensemble_median/videos-for-each-labeled-frame/PRL43_200617_131904_lBack/img00030000_uncropped.csv
Converting img00034980.csv to img00034980_uncropped.csv
Created uncropped file: /team